# Official WebArena-Verified Demo GitLab via uv

Dieses Notebook nutzt das offizielle Repository unter `external/webarena-verified` und bildet den Doku-Workflow mit `uv run invoke -r examples demo-gitlab-start` bzw. `uv run invoke -r examples gitlab-start` ab.

Wichtig: Die Demo-GitLab-Instanz ist zum Nachvollziehen des offiziellen Demo-Flows gedacht. Fuer ein spaeteres Experiment mit vielen echten WebArena-Verified-Aufgaben brauchst du die vollstaendigen WebArena-Instanzen und einen Agenten/Runner.

In [ ]:
from pathlib import Path
import json
import subprocess

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

OFFICIAL_REPO = PROJECT_ROOT / 'external' / 'webarena-verified'
OFFICIAL_REPO

## 1. Voraussetzungen pruefen

In [ ]:
assert OFFICIAL_REPO.exists(), f'Official repo fehlt: {OFFICIAL_REPO}'
subprocess.run(['uv', '--version'], cwd=OFFICIAL_REPO, check=True)
subprocess.run(['docker', '--version'], cwd=OFFICIAL_REPO, check=True)
subprocess.run(['uv', 'run', 'invoke', '-r', 'examples', '--list'], cwd=OFFICIAL_REPO, check=True)

## 2. Demo-GitLab starten

Diese Zelle startet einen Docker-Container `wa-demo-gitlab`. Das kann 2-4 Minuten dauern. Wenn du es lieber im Terminal machen willst, fuehre dort aus:

```bash
cd external/webarena-verified
uv run invoke -r examples gitlab-start
```

In manchen Doku-Versionen steht `demo-gitlab-start`; in diesem Repo heisst der Invoke-Task aktuell `gitlab-start`.

In [ ]:
RUN_GITLAB_START = False

if RUN_GITLAB_START:
    subprocess.run(['uv', 'run', 'invoke', '-r', 'examples', 'gitlab-start'], cwd=OFFICIAL_REPO, check=True)
else:
    print('GitLab-Start uebersprungen. Setze RUN_GITLAB_START = True oder starte im Terminal.')

Nach dem Start erreichbar unter:

- URL: `http://localhost:8012`
- Username: `root`
- Password: `demopass`

## 3. Einzelnen Demo-Task 44 vorbereiten

In [ ]:
output_dir = OFFICIAL_REPO / 'output'
output_dir.mkdir(exist_ok=True)

subprocess.run([
    'uv', 'run', 'webarena-verified', 'agent-input-get',
    '--task-ids', '44',
    '--config', 'examples/configs/config.demo.json',
    '--output', 'output/tasks.demo.json',
], cwd=OFFICIAL_REPO, check=True)

json.loads((output_dir / 'tasks.demo.json').read_text())

## 4. Human-Agent fuer Task 44 starten

Das ist interaktiv und oeffnet einen Browser. Ich wuerde diesen Schritt im Terminal machen:

```bash
cd external/webarena-verified
uv run python examples/agents/human/agent.py \
  --tasks-file output/tasks.demo.json \
  --task_id 44 \
  --task_output_dir output/demo-run/44 \
  --config examples/configs/config.demo.json
```

Danach Browser schliessen und im Terminal die final response bestaetigen.

## 5. Evaluation fuer deinen Human-Run

In [ ]:
RUN_EVAL_TASK_44 = False

if RUN_EVAL_TASK_44:
    subprocess.run([
        'uv', 'run', 'webarena-verified', 'eval-tasks',
        '--config', 'examples/configs/config.demo.json',
        '--task-ids', '44',
        '--output-dir', 'output/demo-run',
    ], cwd=OFFICIAL_REPO, check=True)
else:
    print('Evaluation uebersprungen. Erst Human-Agent-Run erzeugen, dann RUN_EVAL_TASK_44 = True setzen.')

## 6. Zehn GitLab-Agent-Inputs exportieren

Das ist noch kein Loesen von 10 Tasks. Es ist ein leichter Test, ob Task-Daten und URL-Rendering fuer GitLab funktionieren. Die Demo-GitLab-Instanz reicht nicht automatisch fuer alle diese Aufgaben.

In [ ]:
subprocess.run([
    'uv', 'run', 'webarena-verified', 'agent-input-get',
    '--sites', 'gitlab',
    '--config', 'examples/configs/config.demo.json',
    '--output', 'output/gitlab_agent_inputs.json',
], cwd=OFFICIAL_REPO, check=True)

all_gitlab = json.loads((output_dir / 'gitlab_agent_inputs.json').read_text())
first10 = all_gitlab[:10]
(output_dir / 'gitlab_agent_inputs_first10.json').write_text(json.dumps(first10, indent=2))

len(all_gitlab), [task['task_id'] for task in first10]

In [ ]:
first10

## 7. Warum `1,2,3` mit Demo-Config Warnungen erzeugt

`examples/configs/config.demo.json` enthaelt nur `__GITLAB__`. Tasks `1,2,3` gehoeren aber zu `SHOPPING_ADMIN`. Deshalb kann WebArena-Verified diese URLs nicht rendern und schreibt Template-URLs. Das ist kein Docker-Fehler, sondern eine nicht passende Config fuer diese Task-IDs.

In [ ]:
subprocess.run([
    'docker', 'run', '--rm',
    '-v', f'{OFFICIAL_REPO}:/workspace',
    'ghcr.io/servicenow/webarena-verified:latest',
    'agent-input-get',
    '--task-ids', '1,2,3',
    '--config', '/workspace/examples/configs/config.demo.json',
    '--output', '/workspace/output/tasks_shopping_admin_with_demo_config.json',
], cwd=OFFICIAL_REPO, check=True)

json.loads((output_dir / 'tasks_shopping_admin_with_demo_config.json').read_text())

## 8. Vorhandene offizielle Beispiel-Logs evaluieren

Das Repo bringt Beispiel-Logs fuer Tasks 107 und 108 mit. Damit kannst du die Evaluation testen, ohne selbst schon einen Agent-Run erzeugt zu haben.

In [ ]:
subprocess.run([
    'uv', 'run', 'webarena-verified', 'eval-tasks',
    '--config', 'examples/configs/config.demo.json',
    '--task-ids', '107,108',
    '--output-dir', 'examples/agent_logs/demo',
], cwd=OFFICIAL_REPO, check=True)

result_108 = json.loads((OFFICIAL_REPO / 'examples/agent_logs/demo/108/eval_result.json').read_text())
result_108

## 9. Demo-GitLab stoppen

In [ ]:
RUN_GITLAB_STOP = False

if RUN_GITLAB_STOP:
    subprocess.run(['uv', 'run', 'invoke', '-r', 'examples', 'gitlab-stop'], cwd=OFFICIAL_REPO, check=True)
else:
    print('GitLab-Stop uebersprungen. Setze RUN_GITLAB_STOP = True oder stoppe im Terminal.')